# 04. Cortex Agent ― Analyst と Search を組み合わせる

**Cortex Agent** は、質問を理解して**どのツールを使うかを自分で判断し**、結果を回答にまとめる AI エージェントです。

| ツール | 中身 | 得意な質問 |
|---|---|---|
| `sales_ops_analyst`（Cortex Analyst） | 02 の Semantic View v2 | 「何件？」「いくら？」「誰？」 |
| `sales_docs_search`（Cortex Search） | 03 の PDF 検索 | 「推奨は？」「どう対応すべき？」「なぜ？」 |

**両方を使うと**、「レポートの推奨値を、実際のデータが満たしているか」のような質問に 1 回で答えられます。

**前提**：`03_cortex_search.ipynb` を実行済みであること

## Step 0. 実行環境をそろえる

使うロール・ウェアハウス・スキーマを指定します。Notebook はそれぞれ独立して動くため、各 Notebook の最初に実行します。

In [ ]:
%%sql -r setup
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE CORTEX_QUICK_HOL_WH;
USE SCHEMA CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA;
SELECT CURRENT_ROLE() AS "ロール", CURRENT_WAREHOUSE() AS "ウェアハウス", CURRENT_SCHEMA() AS "スキーマ"

## Step 1. Agent を作る

| 設定（SQL の spec の項目） | 内容 |
|---|---|
| `models.orchestration` | 使うモデル。`auto` で Snowflake が自動選択 |
| `instructions.orchestration` | **ツールの使い分けの指示**。数字は Analyst、推奨・方針は Search、比較は両方 |
| `instructions.response` | 回答の書き方。日本語の質問には日本語で答える |
| `tools` / `tool_resources` | 2 つのツールと、それぞれの参照先（Semantic View、Search Service） |

### 方法 A：GUI（AI と ML → Agents）で作る

1. Snowsight 左メニュー **「AI と ML」→「Agents」** を開き、**「Create agent」**
2. データベース / スキーマに `CORTEX_QUICK_HOL_DB` / `CORTEX_QUICK_HOL_SCHEMA`、**Agent object name** に **`SALES_OPS_AGENT`**、**Display name** に `Sales Ops Agent` を入力し、**Create agent**
3. 作成した Agent を開き、**「Edit」**
4. **Tools** → **Cortex Analyst** の **「+ Add」**
   - Name：`sales_ops_analyst`
   - Semantic view：**`SALES_OPS_SEMANTIC_VIEW`**（02 の v2）
   - Warehouse：**`CORTEX_QUICK_HOL_WH`**
   - Description：下の SQL の `sales_ops_analyst` の description を貼り付け → **Add**
5. **Tools** → **Cortex Search Services** の **「+ Add」**
   - Name：`sales_docs_search`
   - Search service：**`SALES_DOCS_SEARCH`**（03）
   - ID column：`CHUNK_ID`、Title column：`FILE_NAME`
   - Description：下の SQL の `sales_docs_search` の description を貼り付け → **Add**
6. **Orchestration**
   - Orchestration model：`auto`
   - Planning instructions：下の SQL の `orchestration:` の文章を貼り付け
   - Response instruction：下の SQL の `response:` の文章を貼り付け（**日本語で答える指示**が入っています）
7. **Save**。右側のチャット欄で「停滞商談は何件？」と聞いて、回答が返ることを確認する

> 💡 ここで画面に入力した内容は、下の SQL の spec と 1 対 1 で対応しています。SQL を見比べると「GUI の設定＝spec の YAML」であることが分かります。

> ℹ️ GUI の手順は Snowflake 公式ドキュメントをもとに書いています。画面のラベルや順番が多少違う場合があります。
> 迷ったら **方法 B（SQL）** のセルを実行すれば、同じ名前・同じ設定で作られます（GUI で作ったものは上書きされます）。

### 方法 B：SQL で作る（`CREATE AGENT`）

In [ ]:
%%sql -r create_agent
CREATE OR REPLACE AGENT CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_AGENT
  COMMENT = 'Sales Operations agent (Cortex Analyst + Cortex Search)'
  FROM SPECIFICATION $$
  models:
    orchestration: auto

  instructions:
    system: >
      You are a Sales Operations analyst. Help users explore revenue, pipeline health,
      sales rep performance, customer segmentation, product mix, and marketing attribution.
    orchestration: >
      Use sales_ops_analyst for numbers from the sales data (revenue, pipeline, reps, customers,
      products, marketing). Use sales_docs_search for recommendations, benchmarks, policies and
      qualitative insights in the internal PDF reports. When a question compares the data with a
      recommendation in the reports, use both tools.
    response: >
      Provide concise, actionable answers. Format USD values as currency and rates as percentages.
      When you use the reports, mention the report name. Always reply in the same language as the
      user's question (reply in Japanese if the question is in Japanese).

  tools:
    - tool_spec:
        type: cortex_analyst_text_to_sql
        name: sales_ops_analyst
        description: >
          Structured sales operations data: revenue, pipeline, rep performance, customers,
          product plans, and marketing attribution. Includes company KPIs such as stalled deals,
          at-risk deals, win rate and pipeline coverage target.
    - tool_spec:
        type: cortex_search
        name: sales_docs_search
        description: >
          Internal PDF reports (Sales Performance Analysis, Customer Intelligence & Pipeline) with
          benchmarks, recommendations, and qualitative insights.

  tool_resources:
    sales_ops_analyst:
      semantic_view: CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_SEMANTIC_VIEW
      execution_environment:
        type: warehouse
        warehouse: CORTEX_QUICK_HOL_WH
    sales_docs_search:
      name: CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_DOCS_SEARCH
      max_results: 4
      title_column: FILE_NAME
      id_column: CHUNK_ID
  $$

GUI・SQL どちらで作った場合も、次のセルで権限を付けます（Quick からはハンズオン用ロールで実行されるため）。Agent を作り直すと権限は消えるので、そのたびに実行してください。

In [ ]:
%%sql -r grant_agent
GRANT USAGE ON AGENT CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_AGENT TO ROLE CORTEX_QUICK_HOL_ROLE

In [ ]:
%%sql -r show_agent_grants
SHOW GRANTS ON AGENT CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_AGENT

## Step 2. Agent に質問する

Quick につなぐ前に、Snowflake 上で答えられるかを確認します。**ここで失敗する場合は、Quick からも同じエラーになります。**
各回答の上に「使ったツール」が表示されます。質問によって Agent がツールを使い分けていることを確かめてください。

`ASK_AGENT` は、Agent を呼び出す関数 `SNOWFLAKE.CORTEX.DATA_AGENT_RUN` の結果（JSON）を、読みやすい表に整形するだけの SQL プロシージャです。
以降は `CALL ASK_AGENT('質問');` の 1 行で Agent に質問できます。表の「種別」で、Agent がどのツールを使い、どんな SQL を作り、どの文書を参照したかが分かります。（02 で作ったものと同じです。02 を飛ばした場合に備えて、ここでも作成します）

In [ ]:
%%sql -r create_ask_agent
-- Agent に質問し、「使ったツール」「生成した SQL」「参照した文書」「回答」を表に整形して返す
CREATE OR REPLACE PROCEDURE CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.ASK_AGENT(
  QUESTION STRING, AGENT STRING DEFAULT 'SALES_OPS_AGENT')
RETURNS TABLE ("種別" STRING, "内容" STRING)
LANGUAGE SQL
EXECUTE AS CALLER
AS
$$
DECLARE
  agent_fqn STRING DEFAULT 'CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.' || AGENT;
  body STRING DEFAULT OBJECT_CONSTRUCT('messages', ARRAY_CONSTRUCT(OBJECT_CONSTRUCT(
         'role', 'user', 'content', ARRAY_CONSTRUCT(OBJECT_CONSTRUCT('type', 'text', 'text', QUESTION)))))::STRING;
  res RESULTSET;
BEGIN
  res := (
    WITH r AS (SELECT TRY_PARSE_JSON(SNOWFLAKE.CORTEX.DATA_AGENT_RUN(:agent_fqn, :body)) AS J),
         c AS (SELECT f.index AS N, f.value AS V FROM r, LATERAL FLATTEN(J:content) f)
    SELECT CASE V:type::STRING WHEN 'text' THEN '回答' WHEN 'tool_use' THEN 'ツール' ELSE '参照文書' END,
           CASE V:type::STRING
             WHEN 'text'     THEN V:text::STRING
             WHEN 'tool_use' THEN V:tool_use:name::STRING || IFF(V:tool_use:input:sql IS NULL, '', CHR(10) || V:tool_use:input:sql::STRING)
             ELSE ARRAY_TO_STRING(ARRAY_DISTINCT(TRANSFORM(V:tool_result:content[0]:json:search_results, x -> x:doc_title::STRING)), ', ')
           END
    FROM c
    WHERE V:type::STRING IN ('text', 'tool_use')
       OR (V:type::STRING = 'tool_result' AND V:tool_result:content[0]:json:search_results IS NOT NULL)
    ORDER BY N
  );
  RETURN TABLE(res);
END;
$$;
GRANT USAGE ON PROCEDURE CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.ASK_AGENT(STRING, STRING) TO ROLE CORTEX_QUICK_HOL_ROLE

| # | 質問 | 使うはずのツール | 期待される結果（目安） |
|---|---|---|---|
| 1 | 地域別の売上を教えて | Analyst | 北米 約 $5.95M が最大 |
| 2 | 停滞商談は何件で、金額はいくら？ | Analyst | 43 件、約 $4.37M |
| 3 | レポートで推奨されているパイプラインカバレッジは？ | Search | 1.6 倍（Sales Performance Analysis） |
| 4 | 推奨カバレッジを下回っている営業担当は何人？推奨値の根拠も教えて | **両方** | 63 人、1.6 倍の根拠 |
| 5 | 停滞商談への推奨アクションは？対象の件数と金額も | **両方** | 役員スポンサーの関与など＋43 件 |

> 💡 1 回の質問には 15〜40 秒ほどかかります。Quick から呼ぶ場合も同じです（60 秒を超えるとタイムアウト）。

In [ ]:
%%sql -r ask_q1
CALL CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.ASK_AGENT('地域別の売上を教えて')

In [ ]:
%%sql -r ask_q2
CALL CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.ASK_AGENT('停滞商談は何件で、金額はいくら？')

In [ ]:
%%sql -r ask_q3
CALL CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.ASK_AGENT('レポートで推奨されているパイプラインカバレッジは？')

In [ ]:
%%sql -r ask_q4
CALL CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.ASK_AGENT('推奨カバレッジを下回っている営業担当は何人？推奨値の根拠も教えて')

In [ ]:
%%sql -r ask_q5
CALL CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.ASK_AGENT('停滞商談への推奨アクションは？対象の件数と金額も')

### ハンズオン用ロールで実行できるか確認する

Quick からはハンズオン用ロールで実行されます。ロールを切り替えて同じように答えられるか確かめます。

In [ ]:
%%sql -r ask_as_hol_role_1
USE ROLE CORTEX_QUICK_HOL_ROLE;
CALL CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.ASK_AGENT('営業担当の目標達成率トップ3は？')

In [ ]:
%%sql -r ask_as_hol_role_2
CALL CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.ASK_AGENT('レポートで推奨されているパイプラインカバレッジは？')

確認できたら、ロールを ACCOUNTADMIN に戻します。

In [ ]:
%%sql -r back_to_admin
USE ROLE ACCOUNTADMIN

## Step 3. Agent Studio で確認・編集する（GUI）

作った Agent は、Snowsight の画面から確認・編集できます（方法 A で作った場合は、作成時の画面と同じです）。

1. Snowsight 左メニュー **「AI と ML」→「Agents」**（Agent Studio）を開く
2. `SALES_OPS_AGENT` をクリック
3. **「Configuration」** タブで次を確認する
   - **Tools**：`sales_ops_analyst`（Semantic View）と `sales_docs_search`（Cortex Search）の 2 つ
   - **Instructions**：ツールの使い分けの指示
4. 右側のチャット欄で、上の質問を試す。回答の下の **「思考過程（Thinking）」** を開くと、どのツールをどの順に呼んだかが分かります
5. （任意）**Sample questions** に質問例を追加し、右上の **「Save」** をクリック

> ⚠️ GUI で編集した後に、このNotebookの `create_agent` セルを再実行すると、**GUI の変更は上書きされます**。

> 💡 **Snowflake Intelligence でも使えます**：Snowsight 左メニュー **「AI と ML」→「Snowflake Intelligence」** で `SALES_OPS_AGENT` を選ぶと、チャット画面から同じ Agent に質問できます（02 で `use_as_onboarding_question: true` にした質問が候補に出ます）。

> ✅ すべての質問に回答が返り、質問 4・5 で 2 つのツールが使われていれば成功です。

次は `05_mcp_and_oauth.ipynb` に進みます。